# 手撕 Adam / AdamW 优化器

## 背景
Adam = Momentum（一阶矩）+ RMSProp（二阶矩），自适应学习率。
AdamW 修正了 Adam 中 weight decay 的正则化错误：Adam 将 wd 混入梯度（L2 正则），AdamW 直接衰减参数（解耦正则）。

## 考察点
- 偏置修正（bias correction）的原理与为何需要
- Adam vs AdamW 的区别（decoupled weight decay）
- 数值稳定性（eps 的作用）

In [ ]:
import torch

class Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = list(params)
        self.lr, self.beta1, self.beta2, self.eps, self.wd = lr, betas[0], betas[1], eps, weight_decay
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None: continue
            g = p.grad + (self.wd * p if self.wd > 0 else 0)  # L2 正则混入梯度
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

class AdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0):
        self.params = list(params)
        self.lr, self.beta1, self.beta2, self.eps, self.wd = lr, betas[0], betas[1], eps, weight_decay
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None: continue
            if self.wd > 0:
                p.data -= self.lr * self.wd * p.data  # 解耦 weight decay
            g = p.grad
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

In [ ]:
# 在 f(x,y) = x^2 + 10y^2 上验证收敛
torch.manual_seed(42)
p_adam = torch.tensor([5.0, 5.0], requires_grad=True)
p_adamw = torch.tensor([5.0, 5.0], requires_grad=True)
opt_adam = Adam([p_adam], lr=0.1, weight_decay=0.01)
opt_adamw = AdamW([p_adamw], lr=0.1, weight_decay=0.01)
for _ in range(200):
    for p, opt in [(p_adam, opt_adam), (p_adamw, opt_adamw)]:
        loss = p[0]**2 + 10 * p[1]**2
        if p.grad: p.grad.zero_()
        loss.backward()
        opt.step()
print(f"Adam  最终: {p_adam.data.tolist()}")
print(f"AdamW 最终: {p_adamw.data.tolist()}")
assert p_adam.norm() < 0.5 and p_adamw.norm() < 0.5
print("✅ Adam 和 AdamW 均收敛到原点附近")